# Kyoto Clinic Reviews — GPT-5.6 Sol Batch Sentiment Classification

**Input:** `Kyoto_clinics.xlsx`  
**Output:** `Kyoto_clinics_GPT56Sol_final.xlsx`

This notebook is the quantitative sentiment-preparation strand of a **mixed-methods clinic-review study**.

### Methodological separation
- Only `review_text` is sent to the model.
- `review_rating`, clinic name, author information, links, and MAXQDA codes are **never included in the sentiment-classification request**.
- The GPT task is deliberately limited to **overall sentiment** (`negative`, `neutral`, `positive`).
- Qualitative themes are coded independently in MAXQDA. The model is **not** asked to generate themes, so the qualitative strand remains analytically distinct.
- Star ratings are retained as an independent ordinal measure for later triangulation; they are not treated as sentiment ground truth.

A deterministic sentiment index is derived locally only for aggregation:
`negative = -1`, `neutral = 0`, `positive = +1`.


In [ ]:
# Run once if needed
%pip install -U openai pandas openpyxl


Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import os
import json
import hashlib
import pandas as pd


In [ ]:
# ---------------- Configuration ----------------
DATA_FILE = Path("Kyoto_clinics.xlsx")

MODEL = "gpt-5.6-sol"
ENDPOINT = "/v1/responses"

BATCH_INPUT_FILE = Path("kyoto_clinics_gpt56_sol_sentiment_batch.jsonl")
MANIFEST_FILE = Path("kyoto_clinics_gpt56_sol_batch_manifest.csv")
BATCH_META_FILE = Path("kyoto_clinics_gpt56_sol_batch_metadata.json")
RAW_OUTPUT_FILE = Path("kyoto_clinics_gpt56_sol_batch_output.jsonl")
RESULTS_FILE = Path("kyoto_clinics_gpt56_sol_sentiment_results.csv")
FAILURES_FILE = Path("kyoto_clinics_gpt56_sol_batch_failures.csv")
MERGED_FILE = Path("Kyoto_clinics_GPT56Sol_final.xlsx")

EXPECTED_ROWS = 4_086
EXPECTED_TEXT_REVIEWS = 3_981

PROMPT = (
    "Classify the overall sentiment expressed in this Google Maps review text about "
    "a clinic, hospital, medical center, or other healthcare service in Kyoto Prefecture, Japan. "
    "Use only the review text. "
    "positive = predominantly favorable opinion, praise, satisfaction, gratitude, trust, or recommendation; "
    "negative = predominantly unfavorable opinion, criticism, dissatisfaction, frustration, complaint, or distrust; "
    "neutral = mainly factual or descriptive with no clear polarity, or genuinely mixed with no dominant polarity. "
    "If both positive and negative experiences appear, choose the dominant overall sentiment. "
    "Return only the required structured output."
)

SENTIMENT_SCHEMA = {
    "type": "object",
    "properties": {
        "sentiment": {
            "type": "string",
            "enum": ["negative", "neutral", "positive"],
        }
    },
    "required": ["sentiment"],
    "additionalProperties": False,
}

SCORE_MAP = {"negative": -1, "neutral": 0, "positive": 1}

print("Data file:", DATA_FILE)
print("Model:", MODEL)


Data file: Kyoto_clinics.xlsx
Model: gpt-5.6-sol


## 1. Load and validate the clinic dataset

The source workbook should contain 4,086 review rows. Only rows with non-empty `review_text` are eligible for model inference.

The notebook also checks that `review_id` is unique and sequential. The star rating is retained locally but is not sent to OpenAI.


In [ ]:
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"{DATA_FILE} was not found. Put it in the same folder as this notebook."
    )

df = pd.read_excel(DATA_FILE, engine="openpyxl").copy()

required = {"review_id", "clinic", "review_text", "review_rating"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

ids = pd.to_numeric(df["review_id"], errors="raise").astype(int)

if len(df) != EXPECTED_ROWS:
    raise ValueError(f"Expected {EXPECTED_ROWS:,} rows; found {len(df):,}.")
if not ids.is_unique:
    raise ValueError("review_id must be unique.")
if ids.tolist() != list(range(1, EXPECTED_ROWS + 1)):
    raise ValueError(
        f"review_id must be sequential 1..{EXPECTED_ROWS:,}."
    )

df["review_text_clean"] = df["review_text"].astype("string").str.strip()
eligible = df["review_text_clean"].notna() & df["review_text_clean"].ne("")
api_df = df.loc[eligible, ["review_id", "review_text_clean"]].copy()

if len(api_df) != EXPECTED_TEXT_REVIEWS:
    raise ValueError(
        f"Expected {EXPECTED_TEXT_REVIEWS:,} usable text reviews; "
        f"found {len(api_df):,}."
    )

print(f"Total rows:                 {len(df):,}")
print(f"Rows with usable text:      {len(api_df):,}")
print(f"Rows without usable text:   {(~eligible).sum():,}")
print(f"Clinics:                    {df['clinic'].nunique():,}")
print()
print("Star-rating distribution:")
display(df["review_rating"].value_counts().sort_index())


Total rows:                 4,086
Rows with usable text:      3,981
Rows without usable text:   105
Clinics:                    110

Star-rating distribution:


review_rating
1 star     1397
2 stars     272
3 stars     283
4 stars     467
5 stars    1667
Name: count, dtype: int64

## 2. Create one Batch request per review

Each request maps directly to one `review_id`. This prevents cross-review influence and allows safe ID-based merging even if Batch outputs are returned in a different order.


In [ ]:
def make_request(review_id: int, review_text: str) -> dict:
    return {
        "custom_id": f"review_{int(review_id)}",
        "method": "POST",
        "url": ENDPOINT,
        "body": {
            "model": MODEL,
            "reasoning": {"effort": "none"},
            "instructions": PROMPT,
            "input": review_text,
            "text": {
                "format": {
                    "type": "json_schema",
                    "name": "sentiment_result",
                    "strict": True,
                    "schema": SENTIMENT_SCHEMA,
                }
            },
            "max_output_tokens": 32,
            "store": False,
        },
    }

requests = []
manifest_rows = []

for row in api_df.itertuples(index=False):
    review_id = int(row.review_id)
    review_text = str(row.review_text_clean)

    requests.append(make_request(review_id, review_text))
    manifest_rows.append({
        "review_id": review_id,
        "custom_id": f"review_{review_id}",
        "review_text_sha256": hashlib.sha256(
            review_text.encode("utf-8")
        ).hexdigest(),
    })

with BATCH_INPUT_FILE.open("w", encoding="utf-8") as f:
    for req in requests:
        f.write(json.dumps(req, ensure_ascii=False) + "\n")

pd.DataFrame(manifest_rows).to_csv(
    MANIFEST_FILE, index=False, encoding="utf-8-sig"
)

print(f"Created {BATCH_INPUT_FILE} with {len(requests):,} requests")
print(f"Created {MANIFEST_FILE}")
print(f"JSONL size: {BATCH_INPUT_FILE.stat().st_size / (1024**2):.2f} MB")

# Leakage checks: only review_text should carry substantive review information.
first_body = requests[0]["body"]
assert "review_rating" not in first_body
assert "clinic" not in first_body
assert "author_title" not in first_body
assert "maxqda" not in json.dumps(first_body).lower()


Created kyoto_clinics_gpt56_sol_sentiment_batch.jsonl with 3,981 requests
Created kyoto_clinics_gpt56_sol_batch_manifest.csv
JSONL size: 6.13 MB


## 3. Inspect a request before spending money

The review text is hidden in the preview.


In [ ]:
preview = json.loads(json.dumps(requests[0]))
preview["body"]["input"] = "<REVIEW TEXT HIDDEN>"
print(json.dumps(preview, indent=2, ensure_ascii=False))


{
  "custom_id": "review_1",
  "method": "POST",
  "url": "/v1/responses",
  "body": {
    "model": "gpt-5.6-sol",
    "reasoning": {
      "effort": "none"
    },
    "instructions": "Classify the overall sentiment expressed in this Google Maps review text about a clinic, hospital, medical center, or other healthcare service in Kyoto Prefecture, Japan. Use only the review text. positive = predominantly favorable opinion, praise, satisfaction, gratitude, trust, or recommendation; negative = predominantly unfavorable opinion, criticism, dissatisfaction, frustration, complaint, or distrust; neutral = mainly factual or descriptive with no clear polarity, or genuinely mixed with no dominant polarity. If both positive and negative experiences appear, choose the dominant overall sentiment. Return only the required structured output.",
    "input": "<REVIEW TEXT HIDDEN>",
    "text": {
      "format": {
        "type": "json_schema",
        "name": "sentiment_result",
        "strict": true,

## 4. Optional synchronous schema smoke test

Use only a few reviews to verify API access and the response schema. Do not tune the prompt against star ratings or later MAXQDA results.


In [ ]:
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    for req in requests[:3]:
        response = client.responses.create(**req["body"])
        print(req["custom_id"], response.output_text)
else:
    print("Smoke test skipped. Set RUN_SMOKE_TEST=True when ready.")


Smoke test skipped. Set RUN_SMOKE_TEST=True when ready.


## 5. Upload the Batch input file


In [ ]:
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

batch_input = client.files.create(
    file=BATCH_INPUT_FILE.open("rb"),
    purpose="batch",
)

meta = {
    "project": "Kyoto clinic mixed-methods sentiment",
    "model": MODEL,
    "endpoint": ENDPOINT,
    "input_file_id": batch_input.id,
    "batch_id": None,
    "output_file_id": None,
    "error_file_id": None,
}
BATCH_META_FILE.write_text(
    json.dumps(meta, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Uploaded file ID:", batch_input.id)
print("Saved:", BATCH_META_FILE)


Uploaded file ID: file-Tr35ARftcr14haZAtUGJqY
Saved: kyoto_clinics_gpt56_sol_batch_metadata.json


## 6. Create the Batch job

The Batch metadata are stored locally so later cells do not require a hard-coded Batch ID.


In [ ]:
meta = json.loads(BATCH_META_FILE.read_text(encoding="utf-8"))

batch = client.batches.create(
    input_file_id=meta["input_file_id"],
    endpoint=ENDPOINT,
    completion_window="24h",
    metadata={
        "project": "Kyoto clinic mixed-methods sentiment",
        "model": MODEL,
    },
)

meta["batch_id"] = batch.id
BATCH_META_FILE.write_text(
    json.dumps(meta, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Batch ID:", batch.id)
print("Status:", batch.status)


Batch ID: batch_6a791133ef1c8190b0b9567a1f17784d
Status: validating


## 7. Check Batch status


In [ ]:
from openai import OpenAI

client = OpenAI()
meta = json.loads(BATCH_META_FILE.read_text(encoding="utf-8"))

if not meta.get("batch_id"):
    raise RuntimeError("No batch_id in metadata. Run the create-Batch cell first.")

batch = client.batches.retrieve(meta["batch_id"])

meta["output_file_id"] = batch.output_file_id
meta["error_file_id"] = batch.error_file_id
BATCH_META_FILE.write_text(
    json.dumps(meta, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Batch ID:", batch.id)
print("Status:", batch.status)
print("Output file ID:", batch.output_file_id)
print("Error file ID:", batch.error_file_id)
if batch.request_counts:
    print("Request counts:", batch.request_counts)


Batch ID: batch_6a791133ef1c8190b0b9567a1f17784d
Status: validating
Output file ID: None
Error file ID: None
Request counts: BatchRequestCounts(completed=0, failed=0, total=0)


## 8. Download and parse completed Batch results

Results are recovered strictly by `custom_id → review_id`, never by output order.


In [2]:
from openai import OpenAI
from pathlib import Path
import json
import pandas as pd

# ============================================================
# Standalone Batch result retrieval cell
# Safe to run after restarting Jupyter.
# No earlier notebook cells are required.
# ============================================================

# Files created by the earlier Batch workflow
BATCH_META_FILE = Path(
    "kyoto_clinics_gpt56_sol_batch_metadata.json"
)
RAW_OUTPUT_FILE = Path(
    "kyoto_clinics_gpt56_sol_batch_output.jsonl"
)
RESULTS_FILE = Path(
    "kyoto_clinics_gpt56_sol_sentiment_results.csv"
)
FAILURES_FILE = Path(
    "kyoto_clinics_gpt56_sol_batch_failures.csv"
)

EXPECTED_TEXT_REVIEWS = 3_981
VALID_LABELS = {"negative", "neutral", "positive"}

# ------------------------------------------------------------
# 1. Load the saved Batch ID
# ------------------------------------------------------------

if not BATCH_META_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find {BATCH_META_FILE}.\n"
        "Make sure this notebook is running in the same folder "
        "where the Batch metadata file was created."
    )

meta = json.loads(
    BATCH_META_FILE.read_text(encoding="utf-8")
)

batch_id = meta.get("batch_id")

if not batch_id:
    raise RuntimeError(
        f"No batch_id found in {BATCH_META_FILE}."
    )

# ------------------------------------------------------------
# 2. Retrieve current Batch status
# ------------------------------------------------------------

client = OpenAI()

batch = client.batches.retrieve(batch_id)

print("Batch ID:", batch.id)
print("Status:", batch.status)

if batch.request_counts is not None:
    print("Request counts:", batch.request_counts)

# ------------------------------------------------------------
# 3. If not finished, stop safely
# ------------------------------------------------------------

if batch.status != "completed":
    print()
    print("Batch is not completed yet.")
    print("Nothing has been downloaded or overwritten.")
    print("Re-run THIS SAME CELL later.")
else:

    if not batch.output_file_id:
        raise RuntimeError(
            "Batch is completed, but no output_file_id was returned."
        )

    # Save latest IDs back to metadata
    meta["output_file_id"] = batch.output_file_id
    meta["error_file_id"] = batch.error_file_id

    BATCH_META_FILE.write_text(
        json.dumps(
            meta,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    print("Output file ID:", batch.output_file_id)
    print("Error file ID:", batch.error_file_id)

    # --------------------------------------------------------
    # 4. Download raw Batch output
    # --------------------------------------------------------

    response = client.files.content(
        batch.output_file_id
    )

    try:
        raw_bytes = response.read()
    except Exception:
        raw_bytes = response.content

    if isinstance(raw_bytes, bytes):
        raw_text = raw_bytes.decode("utf-8")
    else:
        raw_text = str(raw_bytes)

    RAW_OUTPUT_FILE.write_text(
        raw_text,
        encoding="utf-8",
    )

    print()
    print("Saved raw Batch output:")
    print(RAW_OUTPUT_FILE)

    # --------------------------------------------------------
    # 5. Helper: extract text from Responses API body
    # --------------------------------------------------------

    def extract_output_text(body):
        pieces = []

        for item in body.get("output", []) or []:
            if item.get("type") != "message":
                continue

            for part in item.get("content", []) or []:
                if part.get("type") == "output_text":
                    text = part.get("text")
                    if text:
                        pieces.append(text)

        if not pieces:
            return None

        return "".join(pieces).strip()

    # --------------------------------------------------------
    # 6. Parse each Batch response
    # --------------------------------------------------------

    results = []
    failures = []

    for line_no, line in enumerate(
        raw_text.splitlines(),
        start=1,
    ):
        if not line.strip():
            continue

        try:
            record = json.loads(line)
        except Exception as exc:
            failures.append({
                "line_no": line_no,
                "reason": f"Invalid JSONL record: {exc}",
            })
            continue

        custom_id = record.get("custom_id", "")

        response_obj = record.get("response") or {}
        status_code = response_obj.get("status_code")
        body = response_obj.get("body") or {}

        # Recover review_id from custom_id such as review_123
        try:
            review_id = int(
                custom_id.replace("review_", "")
            )
        except Exception:
            failures.append({
                "line_no": line_no,
                "custom_id": custom_id,
                "reason": "Could not recover review_id",
            })
            continue

        # API-level failure
        if status_code != 200:
            failures.append({
                "line_no": line_no,
                "review_id": review_id,
                "custom_id": custom_id,
                "status_code": status_code,
                "reason": record.get("error"),
            })
            continue

        # Extract model output
        output_text = extract_output_text(body)

        if not output_text:
            failures.append({
                "line_no": line_no,
                "review_id": review_id,
                "custom_id": custom_id,
                "reason": "No output text found",
            })
            continue

        # Structured output should be:
        # {"sentiment": "negative|neutral|positive"}
        try:
            parsed = json.loads(output_text)

            sentiment = str(
                parsed["sentiment"]
            ).strip().lower()

            if sentiment not in VALID_LABELS:
                raise ValueError(
                    f"Unexpected label: {sentiment}"
                )

        except Exception as exc:
            failures.append({
                "line_no": line_no,
                "review_id": review_id,
                "custom_id": custom_id,
                "reason": str(exc),
                "raw_output": output_text,
            })
            continue

        results.append({
            "review_id": review_id,
            "gpt_sentiment": sentiment,
        })

    # --------------------------------------------------------
    # 7. Build and validate result tables
    # --------------------------------------------------------

    results_df = pd.DataFrame(results)

    if len(results_df):
        results_df["review_id"] = pd.to_numeric(
            results_df["review_id"],
            errors="raise",
        ).astype(int)

        results_df = (
            results_df
            .sort_values("review_id")
            .reset_index(drop=True)
        )

    failures_df = pd.DataFrame(failures)

    if (
        len(results_df)
        and results_df["review_id"].duplicated().any()
    ):
        dupes = results_df.loc[
            results_df["review_id"].duplicated(
                keep=False
            ),
            "review_id",
        ].tolist()

        raise ValueError(
            f"Duplicate review IDs in Batch output: {dupes[:20]}"
        )

    # --------------------------------------------------------
    # 8. Save parsed files
    # --------------------------------------------------------

    results_df.to_csv(
        RESULTS_FILE,
        index=False,
        encoding="utf-8-sig",
    )

    failures_df.to_csv(
        FAILURES_FILE,
        index=False,
        encoding="utf-8-sig",
    )

    print()
    print("=" * 60)
    print("BATCH PARSING SUMMARY")
    print("=" * 60)

    print(
        f"Successfully parsed: {len(results_df):,}"
    )
    print(
        f"Parsing failures:    {len(failures_df):,}"
    )
    print(
        f"Expected reviews:    {EXPECTED_TEXT_REVIEWS:,}"
    )

    print()
    print("Saved:")
    print(" ", RESULTS_FILE)
    print(" ", FAILURES_FILE)

    # --------------------------------------------------------
    # 9. Strong completeness checks
    # --------------------------------------------------------

    if len(results_df) != EXPECTED_TEXT_REVIEWS:
        print()
        print(
            "WARNING: Parsed result count does not equal "
            f"{EXPECTED_TEXT_REVIEWS:,}."
        )
        print(
            "Inspect the failures file before proceeding."
        )

    elif len(failures_df) > 0:
        print()
        print(
            "WARNING: There are parsing/API failures."
        )

    else:
        print()
        print(
            "✓ All 3,981 clinic reviews were parsed successfully."
        )
        print(
            "✓ No Batch/API/parsing failures detected."
        )

    # --------------------------------------------------------
    # 10. Sentiment distribution
    # --------------------------------------------------------

    if len(results_df):
        sentiment_summary = (
            results_df["gpt_sentiment"]
            .value_counts()
            .reindex(
                ["negative", "neutral", "positive"],
                fill_value=0,
            )
            .rename_axis("sentiment")
            .to_frame("n")
        )

        sentiment_summary["percent"] = (
            sentiment_summary["n"]
            / sentiment_summary["n"].sum()
            * 100
        ).round(2)

        display(sentiment_summary)

Batch ID: batch_6a791133ef1c8190b0b9567a1f17784d
Status: completed
Request counts: BatchRequestCounts(completed=3981, failed=0, total=3981)
Output file ID: file-EuUuhSeUypRWFe2oT49dA5
Error file ID: None

Saved raw Batch output:
kyoto_clinics_gpt56_sol_batch_output.jsonl

BATCH PARSING SUMMARY
Successfully parsed: 3,981
Parsing failures:    0
Expected reviews:    3,981

Saved:
  kyoto_clinics_gpt56_sol_sentiment_results.csv
  kyoto_clinics_gpt56_sol_batch_failures.csv

✓ All 3,981 clinic reviews were parsed successfully.
✓ No Batch/API/parsing failures detected.


,n,percent
sentiment,,
negative,1793,45.04
neutral,204,5.12
positive,1984,49.84


## 9. Validate and merge GPT predictions into the clinic workbook

The merged file preserves every original row. The 105 rows without review text remain without GPT sentiment.


In [3]:
from pathlib import Path
import pandas as pd

# ============================================================
# Standalone merge + validation cell
# No earlier notebook cells are required.
# ============================================================

DATA_FILE = Path("Kyoto_clinics.xlsx")

RESULTS_FILE = Path(
    "kyoto_clinics_gpt56_sol_sentiment_results.csv"
)

MERGED_FILE = Path(
    "Kyoto_clinics_GPT56Sol_final.xlsx"
)

EXPECTED_ROWS = 4_086
EXPECTED_TEXT_REVIEWS = 3_981

VALID_LABELS = {
    "negative",
    "neutral",
    "positive",
}

# ------------------------------------------------------------
# 1. Check required files
# ------------------------------------------------------------

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find source dataset: {DATA_FILE}"
    )

if not RESULTS_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find GPT result file: {RESULTS_FILE}\n"
        "Run the Batch retrieval/parsing cell first."
    )

# ------------------------------------------------------------
# 2. Load source dataset and GPT results
# ------------------------------------------------------------

df = pd.read_excel(
    DATA_FILE,
    engine="openpyxl",
).copy()

results_df = pd.read_csv(
    RESULTS_FILE,
).copy()

print("Source rows:", f"{len(df):,}")
print("GPT result rows:", f"{len(results_df):,}")

# ------------------------------------------------------------
# 3. Validate source dataset
# ------------------------------------------------------------

required_source_columns = {
    "review_id",
    "review_text",
}

missing_source_columns = (
    required_source_columns - set(df.columns)
)

if missing_source_columns:
    raise ValueError(
        "Missing source columns: "
        + repr(sorted(missing_source_columns))
    )

if len(df) != EXPECTED_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_ROWS:,} source rows, "
        f"but found {len(df):,}."
    )

df["review_id"] = pd.to_numeric(
    df["review_id"],
    errors="raise",
).astype(int)

if df["review_id"].duplicated().any():
    dupes = (
        df.loc[
            df["review_id"].duplicated(keep=False),
            "review_id",
        ]
        .tolist()
    )

    raise ValueError(
        "Duplicate review_id values in source dataset: "
        + repr(dupes[:20])
    )

# ------------------------------------------------------------
# 4. Validate GPT result file
# ------------------------------------------------------------

required_result_columns = {
    "review_id",
    "gpt_sentiment",
}

missing_result_columns = (
    required_result_columns - set(results_df.columns)
)

if missing_result_columns:
    raise ValueError(
        "Missing GPT result columns: "
        + repr(sorted(missing_result_columns))
    )

results_df["review_id"] = pd.to_numeric(
    results_df["review_id"],
    errors="raise",
).astype(int)

if results_df["review_id"].duplicated().any():
    dupes = (
        results_df.loc[
            results_df["review_id"].duplicated(keep=False),
            "review_id",
        ]
        .tolist()
    )

    raise ValueError(
        "Duplicate review_id values found in GPT results: "
        + repr(dupes[:20])
    )

results_df["gpt_sentiment"] = (
    results_df["gpt_sentiment"]
    .astype(str)
    .str.strip()
    .str.lower()
)

invalid = ~results_df[
    "gpt_sentiment"
].isin(VALID_LABELS)

if invalid.any():
    raise ValueError(
        "Unexpected GPT labels: "
        + repr(
            sorted(
                results_df.loc[
                    invalid,
                    "gpt_sentiment",
                ].unique()
            )
        )
    )

# ------------------------------------------------------------
# 5. Identify reviews containing usable text
# ------------------------------------------------------------

text_clean = (
    df["review_text"]
    .astype("string")
    .str.strip()
)

has_text = (
    text_clean.notna()
    & text_clean.ne("")
)

n_text = int(has_text.sum())
n_no_text = int((~has_text).sum())

print()
print(
    "Usable text reviews:",
    f"{n_text:,}",
)
print(
    "No-text reviews:",
    f"{n_no_text:,}",
)

if n_text != EXPECTED_TEXT_REVIEWS:
    raise ValueError(
        f"Expected {EXPECTED_TEXT_REVIEWS:,} usable "
        f"text reviews, but found {n_text:,}."
    )

# ------------------------------------------------------------
# 6. Check GPT coverage exactly
# ------------------------------------------------------------

expected_ids = set(
    df.loc[
        has_text,
        "review_id",
    ].astype(int)
)

returned_ids = set(
    results_df[
        "review_id"
    ].astype(int)
)

missing_ids = (
    expected_ids - returned_ids
)

unexpected_ids = (
    returned_ids - expected_ids
)

print()
print(
    "Expected text reviews:",
    f"{len(expected_ids):,}",
)
print(
    "Returned GPT results:",
    f"{len(returned_ids):,}",
)
print(
    "Missing review IDs:",
    len(missing_ids),
)
print(
    "Unexpected review IDs:",
    len(unexpected_ids),
)

if missing_ids:
    print(
        "First missing IDs:",
        sorted(missing_ids)[:20],
    )

if unexpected_ids:
    print(
        "First unexpected IDs:",
        sorted(unexpected_ids)[:20],
    )

if missing_ids or unexpected_ids:
    raise ValueError(
        "GPT results do not exactly match "
        "the usable-text review IDs."
    )

# ------------------------------------------------------------
# 7. Merge GPT sentiment into original dataset
# ------------------------------------------------------------

merged = df.merge(
    results_df[
        [
            "review_id",
            "gpt_sentiment",
        ]
    ],
    on="review_id",
    how="left",
    validate="one_to_one",
)

# Recalculate has_text AFTER merge so the boolean mask
# is guaranteed to align with merged rows.
merged_text_clean = (
    merged["review_text"]
    .astype("string")
    .str.strip()
)

merged_has_text = (
    merged_text_clean.notna()
    & merged_text_clean.ne("")
)

# ------------------------------------------------------------
# 8. Strong post-merge checks
# ------------------------------------------------------------

if len(merged) != EXPECTED_ROWS:
    raise ValueError(
        "Row count changed after merge."
    )

if not merged.loc[
    merged_has_text,
    "gpt_sentiment",
].isin(VALID_LABELS).all():

    raise ValueError(
        "At least one usable-text review "
        "is missing a valid GPT sentiment."
    )

if merged.loc[
    ~merged_has_text,
    "gpt_sentiment",
].notna().any():

    bad_ids = (
        merged.loc[
            ~merged_has_text
            & merged["gpt_sentiment"].notna(),
            "review_id",
        ]
        .tolist()
    )

    raise ValueError(
        "GPT sentiment unexpectedly exists for "
        "reviews without text: "
        + repr(bad_ids[:20])
    )

# ------------------------------------------------------------
# 9. Save final workbook
# ------------------------------------------------------------

merged.to_excel(
    MERGED_FILE,
    index=False,
    engine="openpyxl",
)

print()
print("=" * 60)
print("FINAL MERGE SUMMARY")
print("=" * 60)

print(
    "Total source reviews:",
    f"{len(merged):,}",
)
print(
    "Reviews with text:",
    f"{merged_has_text.sum():,}",
)
print(
    "Reviews without text:",
    f"{(~merged_has_text).sum():,}",
)
print(
    "GPT predictions:",
    f"{merged['gpt_sentiment'].notna().sum():,}",
)
print(
    "Missing GPT predictions:",
    f"{merged['gpt_sentiment'].isna().sum():,}",
)

print()
print("Saved:")
print(MERGED_FILE)

# ------------------------------------------------------------
# 10. GPT sentiment distribution
# ------------------------------------------------------------

sentiment_summary = (
    merged.loc[
        merged_has_text,
        "gpt_sentiment",
    ]
    .value_counts()
    .reindex(
        [
            "negative",
            "neutral",
            "positive",
        ],
        fill_value=0,
    )
    .rename_axis("sentiment")
    .to_frame("n")
)

sentiment_summary["percent"] = (
    sentiment_summary["n"]
    / sentiment_summary["n"].sum()
    * 100
).round(2)

print()
print("GPT sentiment distribution:")

display(sentiment_summary)

print()
print(
    "✓ Final Kyoto clinic GPT workbook "
    "created successfully."
)

Source rows: 4,086
GPT result rows: 3,981

Usable text reviews: 3,981
No-text reviews: 105

Expected text reviews: 3,981
Returned GPT results: 3,981
Missing review IDs: 0
Unexpected review IDs: 0

FINAL MERGE SUMMARY
Total source reviews: 4,086
Reviews with text: 3,981
Reviews without text: 105
GPT predictions: 3,981
Missing GPT predictions: 105

Saved:
Kyoto_clinics_GPT56Sol_final.xlsx

GPT sentiment distribution:


,n,percent
sentiment,,
negative,1793,45.04
neutral,204,5.12
positive,1984,49.84



✓ Final Kyoto clinic GPT workbook created successfully.


## 10. Usage report

This cell reports token usage from the completed Batch output. It intentionally does **not** hard-code dollar prices; pricing can change. If a cost estimate is needed, multiply these token totals by the current official Batch input/output prices.


In [4]:
from pathlib import Path
import json
import pandas as pd

# ============================================================
# Standalone Batch token-usage summary cell
# No earlier notebook cells are required.
# ============================================================

RAW_OUTPUT_FILE = Path(
    "kyoto_clinics_gpt56_sol_batch_output.jsonl"
)

EXPECTED_RECORDS = 3_981

# ------------------------------------------------------------
# 1. Check that the Batch output exists
# ------------------------------------------------------------

if not RAW_OUTPUT_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find {RAW_OUTPUT_FILE}.\n"
        "Run the Batch retrieval/parsing cell first."
    )

# ------------------------------------------------------------
# 2. Accumulate token usage
# ------------------------------------------------------------

total_input = 0
total_output = 0
total_reasoning = 0
total_cached_input = 0

n_records = 0
n_with_usage = 0
n_without_usage = 0

with RAW_OUTPUT_FILE.open(
    "r",
    encoding="utf-8",
) as f:

    for line_no, line in enumerate(f, start=1):

        if not line.strip():
            continue

        try:
            record = json.loads(line)
        except Exception as exc:
            raise ValueError(
                f"Invalid JSON on line {line_no}: {exc}"
            )

        n_records += 1

        response_obj = record.get("response") or {}
        body = response_obj.get("body") or {}
        usage = body.get("usage") or {}

        if not usage:
            n_without_usage += 1
            continue

        n_with_usage += 1

        # Main token counts
        total_input += (
            usage.get("input_tokens", 0) or 0
        )

        total_output += (
            usage.get("output_tokens", 0) or 0
        )

        # Output-token details
        output_details = (
            usage.get("output_tokens_details") or {}
        )

        total_reasoning += (
            output_details.get("reasoning_tokens", 0)
            or 0
        )

        # Input-token details, if available
        input_details = (
            usage.get("input_tokens_details") or {}
        )

        total_cached_input += (
            input_details.get("cached_tokens", 0)
            or 0
        )

# ------------------------------------------------------------
# 3. Derived totals
# ------------------------------------------------------------

total_tokens = (
    total_input
    + total_output
)

non_cached_input = max(
    total_input - total_cached_input,
    0,
)

avg_input = (
    total_input / n_with_usage
    if n_with_usage
    else 0
)

avg_output = (
    total_output / n_with_usage
    if n_with_usage
    else 0
)

avg_reasoning = (
    total_reasoning / n_with_usage
    if n_with_usage
    else 0
)

avg_total = (
    total_tokens / n_with_usage
    if n_with_usage
    else 0
)

# ------------------------------------------------------------
# 4. Summary table
# ------------------------------------------------------------

usage_summary = pd.DataFrame({
    "measure": [
        "batch_records",
        "records_with_usage",
        "records_without_usage",
        "input_tokens",
        "cached_input_tokens",
        "non_cached_input_tokens",
        "output_tokens",
        "reasoning_tokens",
        "total_tokens",
        "avg_input_tokens_per_review",
        "avg_output_tokens_per_review",
        "avg_reasoning_tokens_per_review",
        "avg_total_tokens_per_review",
    ],
    "value": [
        n_records,
        n_with_usage,
        n_without_usage,
        total_input,
        total_cached_input,
        non_cached_input,
        total_output,
        total_reasoning,
        total_tokens,
        avg_input,
        avg_output,
        avg_reasoning,
        avg_total,
    ],
})

# Make counts easier to read
usage_summary["value"] = usage_summary["value"].map(
    lambda x: round(x, 2)
    if isinstance(x, float)
    else x
)

# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

print("=" * 60)
print("GPT-5.6 SOL BATCH TOKEN USAGE")
print("=" * 60)

print(f"Batch records:       {n_records:,}")
print(f"Records with usage:  {n_with_usage:,}")
print(f"Records without:     {n_without_usage:,}")

if n_records != EXPECTED_RECORDS:
    print()
    print(
        f"WARNING: Expected {EXPECTED_RECORDS:,} records "
        f"but found {n_records:,}."
    )
else:
    print(
        f"✓ All {EXPECTED_RECORDS:,} Batch records found."
    )

if n_without_usage > 0:
    print(
        f"WARNING: {n_without_usage:,} records "
        "did not contain usage information."
    )

print()
display(usage_summary)

# ------------------------------------------------------------
# 6. Compact human-readable summary
# ------------------------------------------------------------

print()
print("Token totals")
print("-" * 40)

print(
    "Input tokens:      ",
    f"{total_input:,}",
)

print(
    "  Cached input:    ",
    f"{total_cached_input:,}",
)

print(
    "  Non-cached input:",
    f"{non_cached_input:,}",
)

print(
    "Output tokens:     ",
    f"{total_output:,}",
)

print(
    "Reasoning tokens:  ",
    f"{total_reasoning:,}",
)

print(
    "Total tokens:      ",
    f"{total_tokens:,}",
)

print()
print("Average per review")
print("-" * 40)

print(
    "Input:      ",
    f"{avg_input:,.2f}",
)

print(
    "Output:     ",
    f"{avg_output:,.2f}",
)

print(
    "Reasoning:  ",
    f"{avg_reasoning:,.2f}",
)

print(
    "Total:      ",
    f"{avg_total:,.2f}",
)

GPT-5.6 SOL BATCH TOKEN USAGE
Batch records:       3,981
Records with usage:  3,981
Records without:     0
✓ All 3,981 Batch records found.



,measure,value
0,batch_records,3981.00
1,records_with_usage,3981.00
2,records_without_usage,0.00
3,input_tokens,1011314.00
4,cached_input_tokens,0.00
5,non_cached_input_tokens,1011314.00
6,output_tokens,59715.00
7,reasoning_tokens,0.00
8,total_tokens,1071029.00
9,avg_input_tokens_per_review,254.04



Token totals
----------------------------------------
Input tokens:       1,011,314
  Cached input:     0
  Non-cached input: 1,011,314
Output tokens:      59,715
Reasoning tokens:   0
Total tokens:       1,071,029

Average per review
----------------------------------------
Input:       254.04
Output:      15.00
Reasoning:   0.00
Total:       269.04
